# 📦 SCOR-Py: Panduan Metrik Rantai Pasok (SCOR Level 1 & 2)

Notebook ini memandu Anda menghitung metrik kinerja rantai pasok berbasis kerangka kerja **SCOR (Supply Chain Operations Reference)** menggunakan pustaka **SCOR-Py**.

### Daftar Isi:
1. Persiapan & Import Data
2. Atribut Keandalan (Reliability: POF & OTIF)
3. Atribut Responsivitas (Responsiveness: OFCT)
4. Atribut Biaya (Costs: Total SCM Cost)
5. Atribut Efisiensi Aset (Asset Efficiency: Cash-to-Cash Cycle Time)


In [ ]:
import pandas as pd

from scorpy.asset_efficiency import AssetMetrics
from scorpy.cost import CostMetrics
from scorpy.reliability import ReliabilityMetrics
from scorpy.responsiveness import ResponsivenessMetrics

print("SCOR-Py siap digunakan!")

## 1. Memuat Data Transaksi ERP Sintetis


In [ ]:
df_orders = pd.read_csv("../data/sample_orders.csv")
df_fin = pd.read_csv("../data/sample_financials.csv")
df_inv = pd.read_csv("../data/sample_inventory.csv")

print(f"Total pesanan: {len(df_orders)}")
df_orders.head(3)

## 2. Keandalan: Perfect Order Fulfillment (RL.1.1) & OTIF

Sebuah pesanan dianggap sempurna (**Perfect Order**) jika memenuhi 4 kondisi sekaligus:
1. Tepat Waktu (On-Time)
2. Lengkap (In-Full)
3. Tanpa Kerusakan (Damage-Free)
4. Dokumen Akurat (Doc-Accurate)

Hubungan Invarian: $\text{POF} \le \text{OTIF} \le \min(\text{On-Time}, \text{In-Full})$


In [ ]:
rel = ReliabilityMetrics(
    data=df_orders,
    order_id_col="so_number",
    promised_date_col="promised_delivery_date",
    actual_date_col="actual_delivery_date",
    ordered_qty_col="qty_ordered",
    delivered_qty_col="qty_delivered",
    damaged_flag_col="is_damaged",
    doc_accurate_col="is_doc_complete",
    on_time_grace_days=0,
    in_full_tolerance=0.0,
)

summary = rel.evaluate_all()
for k, v in summary.items():
    print(f"{k:28}: {v}")

### Evaluasi per Segmen Pelanggan


In [ ]:
rel.evaluate_all(group_by="customer_segment")

## 3. Responsivitas: Order Fulfillment Cycle Time (RS.1.1)

Metrik ini mengukur durasi dari pesanan dibuat hingga diterima pelanggan (`actual_delivery_date - order_date`).


In [ ]:
resp = ResponsivenessMetrics(df_orders)
ofct = resp.order_fulfillment_cycle_time()
print("Statistik Waktu Siklus (Hari):")
for k, v in ofct.items():
    print(f"{k:15}: {v} hari")

## 4. Efisiensi Biaya: Total SCM Cost (CO.1.1)

Agregasi biaya transportasi, pergudangan, pengadaan, dan pengembalian barang (*reverse logistics*).


In [ ]:
cost = CostMetrics.from_dataframe(df_fin)
cost_eval = cost.evaluate_all()
print(f"Total Pendapatan : Rp {cost_eval['revenue']:,.0f}")
print(f"Total Biaya SCM  : Rp {cost_eval['total_scm_cost']:,.0f}")
print(f"Rasio Biaya SCM  : {cost_eval['scm_cost_pct_revenue']:.2f}% dari pendapatan")

## 5. Efisiensi Aset: Cash-to-Cash Cycle Time (AM.1.1)

$$\text{C2C} = \text{IDS} + \text{DSO} - \text{DPO}$$


In [ ]:
asset = AssetMetrics.from_dataframes(df_financials=df_fin, df_inventory=df_inv)
c2c = asset.cash_to_cash_cycle()
print(f"IDS (Hari Persediaan) : {c2c['IDS']} hari")
print(f"DSO (Hari Piutang)    : {c2c['DSO']} hari")
print(f"DPO (Hari Utang)      : {c2c['DPO']} hari")
print(f"Cash-to-Cash Net      : {c2c['C2C_days']} hari")